<a href="https://colab.research.google.com/github/nehadid/AAI2025/blob/main/SmartShop_AI_Main_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Project Objective
Build an AI-powered customer service chatbot using
Google Gemini, Python, and LangGraph.

## Three Specialized Agents
1. Order Status Agent
2. Refund and Return Agent
3. Product Recommendation Agent

## Technologies
- Google Gemini API
- Python
- LangGraph
- LangGraph MemorySaver

## Team Responsibilities
Person 1: Main chatbot setup and system prompt

Person 2: Order Status Agent

Person 3: Refund Agent

Person 4: Product Recommendation Agent

Person 5: LangGraph integration and memory

Person 6: Testing and presentation

In [ ]:
%pip install -q -U langchain-google-genai

In [ ]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

api_key = userdata.get("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=api_key
)

response = llm.invoke("Hello! I am building SmartShop AI.")

print(response.content)

Order Status Agent

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

system_prompt = """
You are SmartShop AI, a friendly customer service
assistant for an online shopping store.

Your responsibilities are:
1. Help customers check their order status.
2. Explain refund and return policies.
3. Recommend products based on customer needs.

Always be polite, professional, and helpful.
Use simple and clear language.

Do not invent order details, product information,
or store policies.

Remember previous customer questions when
answering follow-up questions.
"""

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content="Hi! What can you help me with?")
]

response = llm.invoke(messages)

print(response.content)

In [ ]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

# Initialize the LLM
api_key = userdata.get("GOOGLE_API_KEY")
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=api_key
)

system_prompt = """
You are SmartShop AI, a friendly customer service
assistant for an online shopping store.

Your responsibilities are:
1. Help customers check their order status.
2. Explain refund and return policies.
3. Recommend products based on customer needs.

Always be polite, professional, and helpful.
Use simple and clear language.

Do not invent order details, product information,
or store policies.

Remember previous customer questions when
answering follow-up questions.
"""

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(
        content="Hi, I want to check my order status."
    )
]

response = llm.invoke(messages)

print(response.content)

Refund Agent

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

refund_system_prompt = """
You are the Refund and Return Agent for SmartShop AI,
an online shopping customer service chatbot.

Your job is to help customers with:
1. Refund policy questions.
2. Return eligibility.
3. Return instructions.
4. Damaged or defective products.
5. Refund processing times.

SmartShop's SAMPLE refund policy:
- Customers can return items within 30 days of delivery.
- Products must be unused and in original packaging.
- Approved refunds take 5-7 business days to process.
- Customers must provide an order number.
- Damaged or defective products require support review.
- Final-sale products are generally nonreturnable.

Rules:
- Be friendly, polite, and professional.
- Explain policies using simple language.
- Ask for missing information when needed.
- Never invent order information.
- Never claim a refund has been approved or processed.
- If a customer needs special assistance, refer them
  to customer support.
"""


In [ ]:
def refund_agent(state):
    messages = [
        SystemMessage(content=refund_system_prompt)
    ] + state["messages"]

    response = llm.invoke(messages)

    return {"messages": [response]}

In [ ]:
test_state = {
    "messages": [
        HumanMessage(
            content="Hi, I bought a shirt 10 days ago. Can I return it?"
        )
    ]
}

result = refund_agent(test_state)

print(result["messages"][-1].content)

In [ ]:
def check_refund_eligibility(days_since_delivery, used, final_sale=False):

    if final_sale:
        return "Not eligible: This is a final-sale item."

    if days_since_delivery > 30:
        return "Not eligible: The 30-day return window has expired."

    if used:
        return "Not eligible for a standard return: The product has been used."

    return "Potentially eligible: The item meets the basic return conditions."

In [ ]:
print(check_refund_eligibility(10, False))
print(check_refund_eligibility(45, False))
print(check_refund_eligibility(15, True))


In [ ]:
test_questions = [
    "What is your refund policy?",
    "How long does it take to receive my refund?",
    "Can I return an item after 40 days?",
    "I received a damaged product. What should I do?",
    "I lost my order number. Can I still get a refund?"
]

for question in test_questions:
    print("\nCustomer:", question)

    state = {
        "messages": [HumanMessage(content=question)]
    }

    result = refund_agent(state)

    print("Refund Agent:", result["messages"][-1].content)
    print("-" * 50)


Langgraph integration memory

In [ ]:
from typing import Literal
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver

# =========================================================
# 1. Mock Databases
# =========================================================
ORDERS_DB = {
    "12345": {"status": "Shipped", "delivery_date": "2 business days", "carrier": "FedEx", "item": "Shirt"},
    "67890": {"status": "Processing", "delivery_date": "5-7 business days", "carrier": "USPS", "item": "Headphones"},
    "99999": {"status": "Delivered", "delivery_date": "Yesterday", "carrier": "UPS", "item": "Smartwatch"}
}

PRODUCTS = [
    {"name": "Wireless Headphones", "price": 80, "description": "Noise cancellation and long battery life"},
    {"name": "Bluetooth Speaker", "price": 45, "description": "Portable speaker with clear sound"},
    {"name": "Gaming Mouse", "price": 35, "description": "Fast response and comfortable design"},
    {"name": "Running Shoes", "price": 90, "description": "Lightweight and comfortable for running"},
    {"name": "Backpack", "price": 50, "description": "Great for school and travel"},
    {"name": "Smartwatch", "price": 120, "description": "Tracks fitness and notifications"}
]

# Helper function to extract text safely from messages
def extract_text(content) -> str:
    if isinstance(content, str):
        return content
    elif isinstance(content, list):
        return " ".join([item.get("text", "") for item in content if isinstance(item, dict)])
    return str(content)

# =========================================================
# 2. Agent Node Definitions
# =========================================================

# --- Node 1: Order Status Agent ---
order_status_system_prompt = """
You are the Order Status Agent for SmartShop AI.
Your job is to help customers check the status of their orders using our database.
Rules:
- Ask the customer for their order number if they haven't provided one.
- Only report order information found in the database.
- If an order number is not found, politely inform the customer.
- Always be polite, concise, and helpful.
"""

def order_status_agent(state: MessagesState):
    # Safely extract text across string and list message contents
    messages_text = " ".join([extract_text(m.content) for m in state["messages"]])
    context_info = ""
    for order_id, info in ORDERS_DB.items():
        if order_id in messages_text:
            context_info = f"\n[DATABASE LOOKUP SUCCESS]: Order #{order_id} -> Status: {info['status']}, Item: {info['item']}, Delivery: {info['delivery_date']}, Carrier: {info['carrier']}"
            break
    system_msg = SystemMessage(content=order_status_system_prompt + context_info)
    response = llm.invoke([system_msg] + state["messages"])
    return {"messages": [response]}

# --- Node 2: Refund Agent ---
refund_system_prompt = """
You are the Refund and Return Agent for SmartShop AI.
SmartShop Refund Policy:
- Customers can return items within 30 days of delivery.
- Products must be unused and in original packaging.
- Approved refunds take 5-7 business days to process.
- Customers must provide an order number.
- Damaged/defective products require support review.
- Final-sale products are non-returnable.
Rules: Be friendly, polite, explain clearly, and ask for missing information.
"""

def refund_agent(state: MessagesState):
    system_msg = SystemMessage(content=refund_system_prompt)
    response = llm.invoke([system_msg] + state["messages"])
    return {"messages": [response]}

# --- Node 3: Product Recommendation Agent ---
product_system_prompt = """
You are the Product Recommendation Agent for SmartShop AI.
Rules:
- Only recommend products from the available list.
- Stay within the customer's budget.
- Explain why each product is a good choice.
- Keep responses friendly and easy to understand.
"""

def product_recommendation_agent(state: MessagesState):
    product_list = "\n".join([f"{p['name']} - ${p['price']} - {p['description']}" for p in PRODUCTS])
    system_msg = SystemMessage(content=product_system_prompt + f"\nAvailable Products:\n{product_list}")
    response = llm.invoke([system_msg] + state["messages"])
    return {"messages": [response]}

# =========================================================
# 3. Router Function
# =========================================================
def route_intent(state: MessagesState) -> Literal["order_status_agent", "refund_agent", "product_recommendation_agent"]:
    last_message_text = extract_text(state["messages"][-1].content).lower()

    order_keywords = ["order", "status", "track", "package", "where is my", "shipped", "delivery"]
    refund_keywords = ["refund", "return", "policy", "damaged", "defective", "exchange"]

    if any(k in last_message_text for k in order_keywords):
        return "order_status_agent"
    elif any(k in last_message_text for k in refund_keywords):
        return "refund_agent"
    return "product_recommendation_agent"

# =========================================================
# 4. Build LangGraph Workflow & Memory
# =========================================================
workflow = StateGraph(MessagesState)

# Add Nodes
workflow.add_node("order_status_agent", order_status_agent)
workflow.add_node("refund_agent", refund_agent)
workflow.add_node("product_recommendation_agent", product_recommendation_agent)

# Add Routing Logic & Edges
workflow.add_conditional_edges(
    START,
    route_intent,
    {
        "order_status_agent": "order_status_agent",
        "refund_agent": "refund_agent",
        "product_recommendation_agent": "product_recommendation_agent"
    }
)

workflow.add_edge("order_status_agent", END)
workflow.add_edge("refund_agent", END)
workflow.add_edge("product_recommendation_agent", END)

# Compile with Memory Checkpointer
checkpointer = MemorySaver()
app = workflow.compile(checkpointer=checkpointer)

# =========================================================
# 5. Interactive Multi-Agent Test
# =========================================================
config = {"configurable": {"thread_id": "full-system-test-v2"}}

print("=== Turn 1: Product Recommendation ===")
t1 = app.invoke({"messages": [HumanMessage(content="Can you recommend a gift under $50?")]}, config=config)
print("SmartShop AI:", extract_text(t1["messages"][-1].content))

print("\n" + "="*50 + "\n")

print("=== Turn 2: Order Status Check ===")
t2 = app.invoke({"messages": [HumanMessage(content="Where is my order #12345?")]}, config=config)
print("SmartShop AI:", extract_text(t2["messages"][-1].content))

print("\n" + "="*50 + "\n")

print("=== Turn 3: Refund Inquiry (Memory Retention) ===")
t3 = app.invoke({"messages": [HumanMessage(content="Can I return that shirt from order #12345 if it doesn't fit?")]}, config=config)
print("SmartShop AI:", extract_text(t3["messages"][-1].content))

Product Reccomondation

In [ ]:
products = [
    {"name": "Wireless Headphones", "price": 80, "description": "Noise cancellation and long battery life"},
    {"name": "Bluetooth Speaker", "price": 45, "description": "Portable speaker with clear sound"},
    {"name": "Gaming Mouse", "price": 35, "description": "Fast response and comfortable design"},
    {"name": "Running Shoes", "price": 90, "description": "Lightweight and comfortable for running"},
    {"name": "Backpack", "price": 50, "description": "Great for school and travel"},
    {"name": "Smartwatch", "price": 120, "description": "Tracks fitness and notifications"}
]

In [ ]:
product_system_prompt = """
You are the Product Recommendation Agent for SmartShop AI.

Your job is to help customers find products based on
their needs, preferences, and budget.

Rules:
- Only recommend products from the available list.
- Do not make up products or prices.
- Stay within the customer's budget.
- Explain why each product is a good choice.
- If no products match, politely explain.
- Keep responses friendly and easy to understand.
"""

def product_recommendation_agent(state):

    product_list = "\n".join(
        f"{p['name']} - ${p['price']} - {p['description']}"
        for p in products
    )

    messages = [
        SystemMessage(
            content=product_system_prompt +
            "\nAvailable Products:\n" + product_list
        )
    ] + state["messages"]

    response = llm.invoke(messages)

    return {"messages": [response]}

In [ ]:
# Testing Product Recommendation Agent

test_questions = [
    "Can you recommend headphones under $100?",
    "What products do you have for gaming?",
    "What can I buy for under $50?",
    "I want something to track my fitness.",
    "Do you sell laptops?"
]

for question in test_questions:

    print("\nCustomer:", question)

    state = {
        "messages": [HumanMessage(content=question)]
    }

    result = product_recommendation_agent(state)

    print("Product Agent:", result["messages"][-1].content)
    print("-" * 50)